# Professor Recommendation Agent

This project implements an agent to assist prospective graduate students in finding potential professors based on their research interests and application qualifications. The agent utilizes data scraped from a public spreadsheet containing information about university faculty, their research interests, application requirements, and contact details.

## Data Source

The data used in this project is sourced from a Google Spreadsheet:
[https://docs.google.com/spreadsheets/d/1vcEUT_5bXYFQgIzVKpsMQlYmV2xv15VtLXq2rvXZqRk/edit?gid=0#gid=0](https://docs.google.com/spreadsheets/d/1vcEUT_5bXYFQgIzVKpsMQlYmV2xv15VtLXq2rvXZqRk/edit?gid=0#gid=0)

The code first loads this data into a pandas DataFrame.

## Functionalities

The integrated agent combines the following functionalities:

1.  **Professor Recommendation**: Matches user-provided research interests with the 'Research Interests' of professors in the dataset using TF-IDF vectorization and cosine similarity.
2.  **Application Requirements Check**: Checks if a professor's stated requirements are met by a user's qualifications.
3.  **Contact Strategy**: Extracts and presents the preferred method of contact and homepage for the recommended professors.

## Implementation Details

-   **Data Loading and Preprocessing**: The data is loaded from the Google Spreadsheet. Relevant columns ('Research Interests', 'Requirements', 'How to Reach out') are cleaned by handling missing values, converting to lowercase, and removing whitespace.
-   **Research Interest Matching**: TF-IDF (Term Frequency-Inverse Document Frequency) is used to vectorize the research interests. Cosine similarity is then calculated between the user's interests and the professors' interests to find the most similar matches.
-   **Requirements Check**: A basic function is implemented to check if key requirements mentioned by a professor are met by the user's qualifications. This part can be extended for more sophisticated parsing and matching.
-   **Contact Strategy**: The agent retrieves the 'How to Reach out' and 'Homepage' information for the recommended professors.
-   **Integrated Agent**: A main function `professor_agent` orchestrates the above functionalities, taking user interests and qualifications as input and printing the recommendations, requirements status, and contact information.

## How to Use

1.  Ensure you have the necessary libraries installed (pandas, scikit-learn).
2.  Run the code cells in the notebook sequentially to load data, preprocess it, and define the matching and agent functions.
3.  Provide your research interests as a string and your application qualifications as a dictionary to the `professor_agent` function.
4.  The agent will print the top recommended professors, whether you meet their listed requirements (based on the simplified check), and how to contact them.

In [ ]:
import pandas as pd

sheet_url = 'https://docs.google.com/spreadsheets/d/1vcEUT_5bXYFQgIzVKpsMQlYmV2xv15VtLXq2rvXZqRk/edit?gid=0#gid=0'
url_1 = sheet_url.replace('/edit?gid=', '/export?format=csv&gid=')

try:
  df = pd.read_csv(url_1)
  # Display the first 5 rows
  print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
  # Print the column names and their data types
  print(df.info())
except Exception as e:
  print(f"An error occurred: {e}")

| University               | Faculty        | Research Interests                                                                | Notes          | Homepage                             | Positions                 | Requirements                                    | How to Reach out   | @   |
|:-------------------------|:---------------|:----------------------------------------------------------------------------------|:---------------|:-------------------------------------|:--------------------------|:------------------------------------------------|:-------------------|:----|
| Arizona State University | Hua Wei        | Reinforcement Learning, LLM Reasoning, Urban Data Mining                          | 26 Fall        | https://labs.engineering.asu.edu/hw/ | 1-2 PhDs with RA, Interns | TOEFL/IELTS/Duolingo, no GRE                    | Email              | nan |
| Arizona State University | Xiyang Hu      | ML, LLM, Trustworthy, HCI                                                        

# Task
Build an agent that combines the following functionalities based on the data in the spreadsheet at "https://docs.google.com/spreadsheets/d/1vcEUT_5bXYFQgIzVKpsMQlYmV2xv15VtLXq2rvXZqRk/edit?gid=0#gid=0": Professor Recommendation, Application Requirements, Contact Strategy, and Research Interest Matching.

## Data preprocessing


Clean and prepare the relevant columns in the DataFrame, such as 'Research Interests', 'Requirements', and 'How to Reach out', for text processing and analysis.


**Reasoning**:
 need to clean the specified columns by handling missing values, converting to lowercase, and removing whitespace to prepare them for text processing.



In [ ]:
cols_to_clean = ['Research Interests', 'Requirements', 'How to Reach out']

# 1. Handle missing values
for col in cols_to_clean:
    df[col] = df[col].fillna('').astype(str)

# 2. Convert to lowercase and 3. Remove leading/trailing whitespace
for col in cols_to_clean:
    df[col] = df[col].str.lower().str.strip()

# Display the first 5 rows of the cleaned columns
print(df[cols_to_clean].head().to_markdown(index=False, numalign="left", stralign="left"))
# Print the column names and their data types to confirm changes
print(df[cols_to_clean].info())

| Research Interests                                                                | Requirements                                    | How to Reach out   |
|:----------------------------------------------------------------------------------|:------------------------------------------------|:-------------------|
| reinforcement learning, llm reasoning, urban data mining                          | toefl/ielts/duolingo, no gre                    | email              |
| ml, llm, trustworthy, hci                                                         |                                                 | email              |
| cryptography, computer security and privacy                                       | toefl/ietls                                     | email              |
| robotics, robot learning, manipulation, human-robot interaction, control          | toefl or ielts, no gre if you have publications | email              |
| human-ai collaboration, human-robot interaction, and use

## Research interest matching implementation

a mechanism to match user-provided research interests with the 'Research Interests' of professors in the dataset. This could involve techniques like keyword matching or calculating similarity scores between text embeddings.


**Reasoning**:
Import the necessary libraries for TF-IDF vectorization and cosine similarity calculation.



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

**Reasoning**:
Initializing a TfidfVectorizer, fit it to the 'Research Interests' column, and transform the text data into TF-IDF vectors.



In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Research Interests'])

**Reasoning**:
Define a function to match user research interests with the professor research interests using cosine similarity.



In [ ]:
def match_research_interests(user_interests, tfidf_matrix, tfidf_vectorizer):
    """
    Matches user research interests with professor research interests.

    Args:
        user_interests (str): The user's research interests.
        tfidf_matrix (sparse matrix): The TF-IDF matrix of professor research interests.
        tfidf_vectorizer (TfidfVectorizer): The fitted TF-IDF vectorizer.

    Returns:
        numpy.ndarray: An array of cosine similarity scores.
    """
    user_tfidf = tfidf_vectorizer.transform([user_interests])
    similarity_scores = cosine_similarity(user_tfidf, tfidf_matrix)
    return similarity_scores[0]


**Reasoning**:
Define a function to get professor recommendations based on user research interests and similarity scores.

In [ ]:
def get_professor_recommendations(user_interests, df, tfidf_matrix, tfidf_vectorizer, top_n=5):
    """
    Gets top N professor recommendations based on user research interests.

    Args:
        user_interests (str): The user's research interests.
        df (pd.DataFrame): The DataFrame containing professor data.
        tfidf_matrix (sparse matrix): The TF-IDF matrix of professor research interests.
        tfidf_vectorizer (TfidfVectorizer): The fitted TF-IDF vectorizer.
        top_n (int): The number of top recommendations to return.

    Returns:
        pd.DataFrame: A DataFrame containing the top N recommended professors.
    """
    similarity_scores = match_research_interests(user_interests, tfidf_matrix, tfidf_vectorizer)
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    return df.iloc[top_indices]

## Application Requirements Matching Implementation
A function to check if a professor's requirements are met by a user's qualifications. This involves parsing the 'Requirements' column and comparing it against user-provided information.

In [ ]:
def check_requirements_met(professor_requirements, user_qualifications):
    """
    Checks if a professor's requirements are met by a user's qualifications.

    Args:
        professor_requirements (str): The requirements listed by the professor.
        user_qualifications (dict): A dictionary of user qualifications (e.g., {'toefl': True, 'gre': False}).

    Returns:
        bool: True if all listed requirements are met, False otherwise.
    """
    # This is a simplified implementation. More complex parsing and matching
    # would be needed for a real-world scenario.
    professor_requirements = professor_requirements.lower()
    for requirement, met in user_qualifications.items():
        if requirement.lower() in professor_requirements and not met:
            return False
    return True

## Contact Strategy Implementation
Extracting and presenting the contact information or preferred method of contact for the recommended professors from the 'How to Reach out' column.

In [ ]:
def display_contact_strategy(recommended_professors_df):
    """
    Displays the contact information for recommended professors.

    Args:
        recommended_professors_df (pd.DataFrame): DataFrame containing recommended professors.
    """
    print("\n--- Contact Strategy ---")
    for index, row in recommended_professors_df.iterrows():
        print(f"Professor: {row['Faculty']} ({row['University']})")
        print(f"How to Reach out: {row['How to Reach out']}")
        if pd.notna(row['Homepage']) and row['Homepage'] != '':
            print(f"Homepage: {row['Homepage']}")
        print("-" * 20)

## Integrated Agent Implementation
Combining the developed functionalities (Professor Recommendation, Application Requirements, Contact Strategy) into a single, cohesive agent that takes user input and provides relevant outputs.

In [ ]:
def professor_agent(user_interests, user_qualifications, df, tfidf_matrix, tfidf_vectorizer, top_n=5):
    """
    Integrated agent to recommend professors, check requirements, and display contact strategy.

    Args:
        user_interests (str): The user's research interests.
        user_qualifications (dict): A dictionary of user qualifications (e.g., {'toefl': True, 'gre': False}).
        df (pd.DataFrame): The DataFrame containing professor data.
        tfidf_matrix (sparse matrix): The TF-IDF matrix of professor research interests.
        tfidf_vectorizer (TfidfVectorizer): The fitted TF-IDF vectorizer.
        top_n (int): The number of top recommendations to return.

    Returns:
        None: Prints the recommendations, requirements status, and contact information.
    """
    print("--- Professor Recommendations ---")
    recommended_professors = get_professor_recommendations(user_interests, df, tfidf_matrix, tfidf_vectorizer, top_n)
    print(recommended_professors[['University', 'Faculty', 'Research Interests', 'Requirements']].to_markdown(index=False, numalign="left", stralign="left"))

    print("\n--- Requirements Check ---")
    for index, row in recommended_professors.iterrows():
        requirements_met = check_requirements_met(row['Requirements'], user_qualifications)
        print(f"Professor: {row['Faculty']} ({row['University']})")
        print(f"Requirements Met: {requirements_met}")

    display_contact_strategy(recommended_professors)

## Demonstrate Agent Usage
An example of how to use the integrated professor agent with sample user input.

In [ ]:
# Example Usage:
user_research_interests = "reinforcement learning and robotics"
user_application_qualifications = {"toefl": True, "gre": False}

professor_agent(user_research_interests, user_application_qualifications, df, tfidf_matrix, tfidf_vectorizer)

--- Professor Recommendations ---
| University                                  | Faculty       | Research Interests                                                   | Requirements              |
|:--------------------------------------------|:--------------|:---------------------------------------------------------------------|:--------------------------|
| Boston University                           | Xuezhou Zhang | reinforcement learning                                               |                           |
| University of Maryland, College Park        | Kaiqing Zhang | reinforcement learning, game theory, generative ai, agents, robotics |                           |
| University of Houston                       | Jianyi Yang   | trustworthy ai, reinforcement learning, llms                         | toefl or ielts, no gre    |
| College of William & Mary                   | Huajie Shao   | physics-informed ml, reinforcement learning, and generative ai       | toefl 90 no gre 